In [1]:
## 5-fold cross validation (promotion binary classification) psub##
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, cohen_kappa_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE  # Import SMOTE

# Load the data
data_rf = pd.read_csv('all_asperson_gdpconn_undemean.csv')  # Replace with your actual file path

# Preprocessing steps (as before)
data_rf.dropna(inplace=True)
data_rf.replace(r'^\s*$', np.nan, regex=True, inplace=True)
data_rf.dropna(inplace=True)

# Label encode categorical variables
for var in ['birth_city', 'nation', 'education']:
    le = LabelEncoder()
    data_rf[var] = le.fit_transform(data_rf[var].astype(str))

# Convert birthyear to numeric
data_rf['birthyear'] = pd.to_datetime(data_rf['birthyear'], errors='coerce').dt.year

# Identify unique units (name + birthyear)
data_rf['unique_unit'] = data_rf['name'] + '_' + data_rf['birthyear'].astype(str)

# Calculate the number of unique ranks for each person
promoted_units = (
    data_rf.groupby('unique_unit')['rank']
    .nunique()
    .reset_index()
    .rename(columns={'rank': 'unique_ranks'})
)
promoted_units['promoted'] = (promoted_units['unique_ranks'] > 1).astype(int)

# Merge the 'promoted' status back to the original data
data_rf = data_rf.merge(promoted_units[['unique_unit', 'promoted']], on='unique_unit')

# **Keep only one row per person**
data_rf = data_rf.sort_values(by='rank').drop_duplicates(subset=['unique_unit'])

# Define independent and control variables
independent_vars = ['attractiveness', 'trustworthiness', 'competence', 'aggressiveness']
control_vars = ['birthyear', 'latitude', 'longitude', 'eth_han', 'edu', 'female', 'weighted_avg_gdp_growth_rate', 'weighted_avg_fiscal_growth_rate', 'connections']
dependent_var = 'promoted'

# **Step 1: Define the dependent and independent variables**
X = data_rf[independent_vars + control_vars]
y = data_rf[dependent_var]

# **Step 2: Apply SMOTE to handle class imbalance**
X_resampled, y_resampled = SMOTE(random_state=2024).fit_resample(X, y)

# **Step 3: Define the model**
rf_model = RandomForestClassifier(n_estimators=200, random_state=2024)

# **Step 4: Define a KFold cross-validator**
kf = KFold(n_splits=5, shuffle=True, random_state=2024)

# **Step 5: Define scorers**
scorers = {
    'Accuracy': make_scorer(accuracy_score),
    'Weighted Kappa': make_scorer(cohen_kappa_score, weights='quadratic'),
    'ROC AUC': make_scorer(roc_auc_score)
}

# **Step 6: Perform cross-validation using resampled data**
cv_results = cross_validate(rf_model, X_resampled, y_resampled, cv=kf, scoring=scorers, return_estimator=True)

# **Step 7: Calculate average scores**
avg_accuracy = np.mean(cv_results['test_Accuracy'])
avg_kappa = np.mean(cv_results['test_Weighted Kappa'])
avg_roc_auc = np.mean(cv_results['test_ROC AUC'])

print(f'Average Accuracy: {avg_accuracy:.3f}')
print(f'Average Weighted Kappa: {avg_kappa:.3f}')
print(f'Average ROC AUC: {avg_roc_auc:.3f}')
print(f'Number of unique units (same name and birthyear): {promoted_units.shape[0]}')

# **Feature Importance Analysis**
feature_importances = np.zeros(len(independent_vars + control_vars))

for estimator in cv_results['estimator']:
    feature_importances += estimator.feature_importances_

# **Step 8: Average feature importances**
feature_importances /= kf.n_splits

# **Step 9: Create a DataFrame for feature importances**
features_df = pd.DataFrame({
    'Feature': independent_vars + control_vars,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# **Step 10: Save the feature importances to an Excel file**
output_file_path = 'promotion_results.csv'  # Update this path
features_df.to_csv(output_file_path, index=False)

print(features_df)

Average Accuracy: 0.859
Average Weighted Kappa: 0.718
Average ROC AUC: 0.859
Number of unique units (same name and birthyear): 1420
                            Feature  Importance
2                        competence    0.114199
1                   trustworthiness    0.111192
3                    aggressiveness    0.107013
0                    attractiveness    0.105508
10     weighted_avg_gdp_growth_rate    0.104773
6                         longitude    0.100640
11  weighted_avg_fiscal_growth_rate    0.095260
4                         birthyear    0.093742
5                          latitude    0.087783
12                      connections    0.051692
8                               edu    0.013134
7                           eth_han    0.010322
9                            female    0.004741


In [2]:
## 5-fold cross validation (purge binary classification) psub##
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, cohen_kappa_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE  # Import SMOTE

# Load the data
data_rf = pd.read_csv('all_asperson_gdpconn_undemean.csv')  # Replace with your actual file path

# Preprocessing steps (as before)
data_rf.dropna(inplace=True)
data_rf.replace(r'^\s*$', np.nan, regex=True, inplace=True)
data_rf.dropna(inplace=True)

# Label encode categorical variables
for var in ['birth_city', 'nation', 'education']:
    le = LabelEncoder()
    data_rf[var] = le.fit_transform(data_rf[var].astype(str))

# Convert birthyear to numeric
data_rf['birthyear'] = pd.to_datetime(data_rf['birthyear'], errors='coerce').dt.year

# **Step 1: Define unique units (name + birthyear)**
data_rf['unique_unit'] = data_rf['name'] + '_' + data_rf['birthyear'].astype(str)

# **Step 2: Drop duplicate observations for each unique person (keep only the first row)**
data_rf = data_rf.sort_values(by='rank').drop_duplicates(subset=['unique_unit'])

# **Step 3: Update the count of unique units after duplicates are dropped**
unique_units = data_rf['unique_unit'].nunique()

# Define independent and control variables
independent_vars = ['attractiveness', 'trustworthiness', 'competence', 'aggressiveness']
control_vars = ['birthyear', 'latitude', 'longitude', 'eth_han', 'edu', 'female', 'weighted_avg_gdp_growth_rate', 'weighted_avg_fiscal_growth_rate', 'connections']
dependent_var = 'purge'

# **Step 4: Define the dependent and independent variables**
X = data_rf[independent_vars + control_vars]
y = data_rf[dependent_var]

# **Step 5: Apply SMOTE to handle class imbalance**
X_resampled, y_resampled = SMOTE(random_state=2023).fit_resample(X, y)

# **Step 6: Define the model**
rf_model = RandomForestClassifier(n_estimators=200, random_state=2023)

# **Step 7: Define a KFold cross-validator**
kf = KFold(n_splits=5, shuffle=True, random_state=2023)

# **Step 8: Define scorers**
scorers = {
    'Accuracy': make_scorer(accuracy_score),
    'Weighted Kappa': make_scorer(cohen_kappa_score, weights='quadratic'),
    'ROC AUC': make_scorer(roc_auc_score)
}

# **Step 9: Perform cross-validation using resampled data**
cv_results = cross_validate(rf_model, X_resampled, y_resampled, cv=kf, scoring=scorers, return_estimator=True)

# **Step 10: Calculate average scores**
avg_accuracy = np.mean(cv_results['test_Accuracy'])
avg_kappa = np.mean(cv_results['test_Weighted Kappa'])
avg_roc_auc = np.mean(cv_results['test_ROC AUC'])

print(f'Average Accuracy: {avg_accuracy:.3f}')
print(f'Average Weighted Kappa: {avg_kappa:.3f}')
print(f'Average ROC AUC: {avg_roc_auc:.3f}')
print(f'Number of unique units (same name and birthyear): {unique_units}')

# **Feature Importance Analysis**
feature_importances = np.zeros(len(independent_vars + control_vars))

for estimator in cv_results['estimator']:
    feature_importances += estimator.feature_importances_

# **Step 11: Average feature importances**
feature_importances /= kf.n_splits

# **Step 12: Create a DataFrame for feature importances**
features_df = pd.DataFrame({
    'Feature': independent_vars + control_vars,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# **Step 13: Save the feature importances to an Excel file**
output_file_path = 'purge_results.csv'  # Update this path
features_df.to_csv(output_file_path, index=False)

print(features_df)

Average Accuracy: 0.872
Average Weighted Kappa: 0.743
Average ROC AUC: 0.872
Number of unique units (same name and birthyear): 1420
                            Feature  Importance
11  weighted_avg_fiscal_growth_rate    0.157980
10     weighted_avg_gdp_growth_rate    0.119970
5                          latitude    0.107160
4                         birthyear    0.100943
3                    aggressiveness    0.098261
6                         longitude    0.093303
2                        competence    0.090503
0                    attractiveness    0.089430
1                   trustworthiness    0.079636
12                      connections    0.027889
9                            female    0.017367
8                               edu    0.012357
7                           eth_han    0.005201
